In [1]:
!pip install chromadb openai pandas python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 18.2 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 15.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 9.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.0 MB/s  0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.0
    Uninstalling protobuf-7.34.0:
      Successfully uninstalled protobuf-7.34.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27/27 [chromadb]/27 [chromadb]etry-sdk]protos]


In [27]:
# ──────────────────────────────────────────────
# STEP 1 – Imports and Config
# ──────────────────────────────────────────────

import os
import json
import time
from pathlib import Path

import pandas as pd
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

In [28]:
# ──────────────────────────────────────────────
# STEP 2 – Paths  ## Replace with your actual paths if needed
# ──────────────────────────────────────────────

BASE_DIR = Path(".")
M1_OUTPUT_DIR = BASE_DIR / "milestone1_outputs"
JSONL_PATH = M1_OUTPUT_DIR / "tagged_chunks.jsonl"

M2_OUTPUT_DIR = BASE_DIR / "milestone2_outputs"
M2_OUTPUT_DIR.mkdir(exist_ok=True)

CHROMA_DIR = M2_OUTPUT_DIR / "chroma_db"
CHROMA_DIR.mkdir(exist_ok=True)

print("Milestone 1 JSONL:", JSONL_PATH.resolve())
print("Milestone 2 output dir:", M2_OUTPUT_DIR.resolve())
print("Chroma dir:", CHROMA_DIR.resolve())

Milestone 1 JSONL: /Users/claycasani/PyCharmProjects/BTE440/legal-ai-capstone/notebooks/milestone1_outputs/tagged_chunks.jsonl
Milestone 2 output dir: /Users/claycasani/PyCharmProjects/BTE440/legal-ai-capstone/notebooks/milestone2_outputs
Chroma dir: /Users/claycasani/PyCharmProjects/BTE440/legal-ai-capstone/notebooks/milestone2_outputs/chroma_db


In [29]:
# ──────────────────────────────────────────────
# STEP 3 – Load Tagged Chunk Corpus
# ──────────────────────────────────────────────

chunks = []

with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"Loaded {len(chunks)} chunks.")

# Preview one chunk
print(json.dumps(chunks[0], indent=2)[:1200])

Loaded 260 chunks.
{
  "chunk_id": "Attachment 2 - Sample Master Service Agreement.pdf::chunk_0",
  "chunk_text": "** This is only Mercy Corps\u2019 standard template for master service agreement contract.. Some terms might be added or removed based on the nature of the provided service and its value. If any inconsistency was found between the terms in this template and the ones in the RFP, the RFP terms and PAGE 1 of 11 MASTER SERVICE AGREEMENT Contract No. _______ THIS MASTER SERVICES AGREEMENT entered into as of __________ by and between MERCY CORPS, a State of Washington, U.S.A. nonprofit corporation having its principal office in Portland, Oregon, U.S.A. (\u201cMercy Corps\u201d) and _____________________________ (\u201cContractor\u201d) is as follows: 1. Master Agreement; Specific Services. From time to time, Mercy Corps may request services from Contractor. For each occasion on which Contractor is willing to provide requested services, the parties will enter into a task order (\

In [30]:
# ──────────────────────────────────────────────
# STEP 4 – Prepare Documents and Metadata
# ──────────────────────────────────────────────

documents = []
ids = []
metadatas = []

for chunk in chunks:
    documents.append(chunk["chunk_text"])
    ids.append(chunk["chunk_id"])

    metadatas.append({
        "source_document": chunk["source_document"],
        "chunk_index": int(chunk["chunk_index"]),
        "token_count": int(chunk["token_count"]),
        "clause_tags_str": " | ".join(chunk["clause_tags"]) if chunk["clause_tags"] else "",
        "out_of_range": str(chunk["out_of_range"])
    })

print(f"Prepared {len(documents)} documents for embedding/store.")
print("Sample metadata:", metadatas[0])

Prepared 260 documents for embedding/store.
Sample metadata: {'source_document': 'Attachment 2 - Sample Master Service Agreement.pdf', 'chunk_index': 0, 'token_count': 596, 'clause_tags_str': 'Payment / financial terms | Termination / cancellation', 'out_of_range': 'True'}


In [31]:
# ──────────────────────────────────────────────
# STEP 5A – Load OpenAI API Key (Local)
# ──────────────────────────────────────────────

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized.")

OpenAI client initialized.


In [ ]:
# ──────────────────────────────────────────────
# STEP 5B – Load OpenAI API Key (Colab)
# ──────────────────────────────────────────────

import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = "your_key_here"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [32]:
# ──────────────────────────────────────────────
# STEP 6 – Embedding Function
# ──────────────────────────────────────────────

EMBEDDING_MODEL = "text-embedding-3-small"

def get_embedding(text, model=EMBEDDING_MODEL):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [8]:
# ──────────────────────────────────────────────
# STEP 7 – Generate Embeddings
# ──────────────────────────────────────────────

embeddings = []
start_time = time.time()

for i, doc in enumerate(documents):
    emb = get_embedding(doc)
    embeddings.append(emb)

    if (i + 1) % 25 == 0 or (i + 1) == len(documents):
        print(f"Embedded {i+1}/{len(documents)} chunks")

elapsed = round(time.time() - start_time, 2)
print(f"Embedding complete in {elapsed} seconds.")

Embedded 25/260 chunks
Embedded 50/260 chunks
Embedded 75/260 chunks
Embedded 100/260 chunks
Embedded 125/260 chunks
Embedded 150/260 chunks
Embedded 175/260 chunks
Embedded 200/260 chunks
Embedded 225/260 chunks
Embedded 250/260 chunks
Embedded 260/260 chunks
Embedding complete in 76.15 seconds.


In [33]:
# ──────────────────────────────────────────────
# STEP 8 – Initialize Chroma
# ──────────────────────────────────────────────

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

collection_name = "legal_ai_chunks"

# delete old collection if rerunning cleanly
existing_collections = [c.name for c in chroma_client.list_collections()]
if collection_name in existing_collections:
    chroma_client.delete_collection(collection_name)

collection = chroma_client.create_collection(name=collection_name)

print(f"Created Chroma collection: {collection_name}")

Created Chroma collection: legal_ai_chunks


In [34]:
# ──────────────────────────────────────────────
# STEP 9 – Add Chunks to Chroma
# ──────────────────────────────────────────────

collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings
)

print(f"Added {len(ids)} chunks to Chroma.")

Added 260 chunks to Chroma.


In [35]:
# ──────────────────────────────────────────────
# STEP 10 – Retrieval Function
# ──────────────────────────────────────────────

def retrieve_top_k(query, top_k=5, clause_filter=None, initial_k=15):
    # Get embedding
    query_embedding = get_embedding(query)

    # Query WITHOUT metadata filter
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=initial_k
    )

    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    ids = results.get("ids", [[]])[0]
    distances = results.get("distances", [[]])[0] if "distances" in results else []

    rows = []
    for i in range(len(docs)):
        row = {
            "id": ids[i],
            "document": docs[i],
            "metadata": metas[i],
            "distance": distances[i] if distances else None
        }
        rows.append(row)

    # 🔍 Python-side filtering (SAFE + RELIABLE)
    if clause_filter:
        rows = [
            r for r in rows
            if clause_filter.lower() in r["metadata"].get("clause_tags_str", "").lower()
        ]

    # Take top_k after filtering
    rows = rows[:top_k]

    return {
        "ids": [[r["id"] for r in rows]],
        "documents": [[r["document"] for r in rows]],
        "metadatas": [[r["metadata"] for r in rows]],
        "distances": [[r["distance"] for r in rows]]
    }

In [36]:
# ──────────────────────────────────────────────
# STEP 11 – Pretty Print Retrieval Results
# ──────────────────────────────────────────────

def print_retrieval_results(results, preview_chars=700):
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    ids = results.get("ids", [[]])[0]
    distances = results.get("distances", [[]])[0] if "distances" in results else []

    for i, doc in enumerate(docs):
        print("=" * 100)
        print(f"Rank: {i+1}")
        print(f"Chunk ID: {ids[i]}")
        print(f"Source document: {metas[i].get('source_document')}")
        print(f"Chunk index: {metas[i].get('chunk_index')}")
        print(f"Clause tags: {metas[i].get('clause_tags_str')}")
        if distances:
            print(f"Distance: {distances[i]}")
        print("-" * 100)
        print(doc[:preview_chars])
        print("\n")

In [37]:
# ──────────────────────────────────────────────
# STEP 12 – First Retrieval Test
# ──────────────────────────────────────────────

test_query = "How can either party terminate this agreement?"

results = retrieve_top_k(test_query, top_k=5)

print("Returned IDs:", results["ids"])
print("Returned docs count:", len(results["documents"][0]))

Returned IDs: [['ex10-1.pdf::chunk_2', 'nda2.pdf::chunk_1', 'Cooley SaaS Agreement ACC Form.pdf::chunk_23', 'CONSULTING-AGREEMENT-template-for-1099-ee.pdf::chunk_2', 'Cooley SaaS Agreement ACC Form.pdf::chunk_11']]
Returned docs count: 5


In [38]:
# ──────────────────────────────────────────────
# STEP 13 – Metadata-Filtered Retrieval Test
# ──────────────────────────────────────────────

query = "How can either party terminate this agreement?"
results_filtered = retrieve_top_k(
    query,
    top_k=5,
    clause_filter="Termination / cancellation"
)

print_retrieval_results(results_filtered)

Rank: 1
Chunk ID: ex10-1.pdf::chunk_2
Source document: ex10-1.pdf
Chunk index: 2
Clause tags: Payment / financial terms | Termination / cancellation | IP ownership / licensing | Governing law / jurisdiction | Dispute resolution / arbitration | Warranties / representations
Distance: 0.9471172094345093
----------------------------------------------------------------------------------------------------
in effect for one (1) year. This Agreement shall be automatically renewed for successive one (1) year terms, unless either party sends written notice to the other party at least thirty (30) days before the end of the then-existing term of employment. B. Termination: Either party may terminate this Agreement at any time with thirty (30) days’ written notice to the other party. Upon termination, the Company shall pay the Employee any accrued and unpaid Base Salary and accrued benefits through the termination date. (i) Termination: Either party may terminate this Agreement at any time with thi

In [39]:
# ──────────────────────────────────────────────
# STEP 14 – Sample Query Set
# ──────────────────────────────────────────────

sample_queries = [
    {"query": "How can either party terminate this agreement?", "clause": "termination"},
    {"query": "What happens if confidential information is shared?", "clause": "confidentiality"},
    {"query": "Who owns the intellectual property created?", "clause": "intellectual property"},
    {"query": "What law governs this contract?", "clause": "governing law"},
    {"query": "What are the payment terms?", "clause": "payment"},
    {"query": "What happens in case of breach?", "clause": "liability"},
    {"query": "How are disputes resolved?", "clause": "dispute"},
    {"query": "What are the obligations of each party?", "clause": "obligations"}
]


sample_queries

[{'query': 'How can either party terminate this agreement?',
  'clause': 'termination'},
 {'query': 'What happens if confidential information is shared?',
  'clause': 'confidentiality'},
 {'query': 'Who owns the intellectual property created?',
  'clause': 'intellectual property'},
 {'query': 'What law governs this contract?', 'clause': 'governing law'},
 {'query': 'What are the payment terms?', 'clause': 'payment'},
 {'query': 'What happens in case of breach?', 'clause': 'liability'},
 {'query': 'How are disputes resolved?', 'clause': 'dispute'},
 {'query': 'What are the obligations of each party?', 'clause': 'obligations'}]

In [41]:
# ──────────────────────────────────────────────
# STEP 15 – Baseline Retrieval Logging
# ──────────────────────────────────────────────

import pandas as pd

rows = []

for item in sample_queries:
    query = item["query"]
    clause = item["clause"]

    results = retrieve_top_k(query, top_k=5, clause_filter=clause)

    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    ids = results.get("ids", [[]])[0]
    distances = results.get("distances", [[]])[0]

    for i in range(len(docs)):
        rows.append({
            "query": query,
            "clause_filter": clause,
            "rank": i + 1,
            "chunk_id": ids[i],
            "text": docs[i],
            "distance": distances[i],
            "source_doc": metas[i].get("source_doc"),
            "clause_tags": metas[i].get("clause_tags_str")
        })

df = pd.DataFrame(rows)

print("Total retrieval rows:", len(df))
df.head()

Total retrieval rows: 30


,query,clause_filter,rank,chunk_id,text,distance,source_doc,clause_tags
0,How can either party terminate this agreement?,termination,1,ex10-1.pdf::chunk_2,in effect for one (1) year. This Agreement sha...,0.947779,None,Payment / financial terms | Termination / canc...
1,How can either party terminate this agreement?,termination,2,nda2.pdf::chunk_1,or service. The Parties shall use the Confiden...,0.983552,None,Payment / financial terms | Termination / canc...
2,How can either party terminate this agreement?,termination,3,Cooley SaaS Agreement ACC Form.pdf::chunk_23,"exceeded, either Party may exercise their indi...",0.983844,None,Termination / cancellation | IP ownership / li...
3,How can either party terminate this agreement?,termination,4,CONSULTING-AGREEMENT-template-for-1099-ee.pdf:...,constitutes the exclusive property of the Comp...,0.992577,None,Termination / cancellation | Liability / indem...
4,How can either party terminate this agreement?,termination,5,Cooley SaaS Agreement ACC Form.pdf::chunk_11,in an Order for a particular termination right...,0.998783,None,Payment / financial terms | Termination / canc...


In [44]:
# ──────────────────────────────────────────────
# STEP 16 – Save Retrieval Logs
# ──────────────────────────────────────────────

df.to_csv("milestone2_outputs/retrieval_results.csv", index=False)

import json
with open("milestone2_outputs/retrieval_results.json", "w") as f:
    json.dump(rows, f, indent=2)

print("Saved retrieval_results.csv and retrieval_results.json")

Saved retrieval_results.csv and retrieval_results.json


In [43]:
# ──────────────────────────────────────────────
# STEP 17 – Milestone 2 Summary
# ──────────────────────────────────────────────

print("=== MILESTONE 2 SUMMARY ===")
print("Total chunks indexed:", collection.count())
print("Total queries:", len(sample_queries))
print("Total retrieval rows:", len(df))
print("Top-k per query: 5")

=== MILESTONE 2 SUMMARY ===
Total chunks indexed: 260
Total queries: 8
Total retrieval rows: 30
Top-k per query: 5


In [19]:
print("Collection count:", collection.count())

Collection count: 260
